# Hotel Intelligence Platform — Complete AI/ML System Overview

**Author:** Mehmet Isik  
**Date:** March 2026  
**Repository:** [hotel-intelligence-platform](https://github.com/mehmetisik/hotel-intelligence-platform)  
**License:** MIT

---

## 1. Introduction

### What is the Hotel Intelligence Platform?

An **end-to-end AI/ML system** built for the hospitality industry that transforms raw booking, invoice, and review data into actionable intelligence. The platform covers the full data-science lifecycle — from predictive modeling and NLP to conversational analytics and production monitoring.

### Key Numbers at a Glance

| Metric | Value |
|---|---|
| Total Bookings Processed | **119,209** |
| Unique Customers Modeled | **12,000** |
| ML Models Trained | **5** (Cancellation, CLTV, Segmentation, Sentiment, Aspect) |
| Platform Modules | **4** |
| Streamlit Dashboard Pages | **7** |
| Estimated Revenue Impact | **EUR 2M+** annually |

### Architecture Overview

```
+-------------------------------------------------------------------+
|                   HOTEL INTELLIGENCE PLATFORM                     |
+-------------------------------------------------------------------+
|                                                                   |
|  +-------------------+  +-------------------+                     |
|  | MODULE 1          |  | MODULE 2          |                     |
|  | Predictive        |  | LLM & Unstructured|                     |
|  | Analytics         |  | Data              |                     |
|  |                   |  |                   |                     |
|  | - Cancellation    |  | - Invoice Class.  |                     |
|  | - CLTV            |  | - Fuzzy Matching  |                     |
|  | - Segmentation    |  | - Sentiment Anal. |                     |
|  +-------------------+  +-------------------+                     |
|                                                                   |
|  +-------------------+  +-------------------+                     |
|  | MODULE 3          |  | MODULE 4          |                     |
|  | Conversational AI |  | MLOps             |                     |
|  |                   |  |                   |                     |
|  | - NL-to-SQL       |  | - Drift Detection |                     |
|  | - Intent Detect.  |  | - Model Monitoring|                     |
|  | - LLM Routing     |  | - Alert System    |                     |
|  +-------------------+  +-------------------+                     |
|                                                                   |
|  +---------------------------------------------------+           |
|  |          STREAMLIT PREMIUM DASHBOARD (7 pages)     |           |
|  +---------------------------------------------------+           |
+-------------------------------------------------------------------+
```

---

## 2. Setup

In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

# Project root
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')

print(f"Project root: {PROJECT_ROOT}")
print(f"Models dir:   {MODELS_DIR}")

---

## 3. Module 1 — Predictive Analytics

Module 1 delivers three predictive pipelines trained on 119K hotel booking records:

| Model | Algorithm | Key Metric |
|---|---|---|
| Cancellation Prediction | XGBoost | AUC-ROC **0.9467** |
| Customer Lifetime Value | BG/NBD + Gamma-Gamma | 6-month CLTV for 12K customers |
| Customer Segmentation | KMeans + Classifier | Classifier accuracy **96.75%** |

### 3.1 Cancellation Prediction — Best Model Demo

In [ ]:
import joblib
import pandas as pd
import numpy as np

# Load the best model and metadata
cancel_dir = os.path.join(MODELS_DIR, 'cancellation')
model = joblib.load(os.path.join(cancel_dir, 'best_model.joblib'))

with open(os.path.join(cancel_dir, 'metadata.json')) as f:
    meta = json.load(f)

print(f"Best model : {meta['best_model']}")
print(f"AUC-ROC    : {meta['best_auc_roc']}")
print(f"Features   : {meta['feature_count']}")
print(f"Train size : {meta['train_size']:,}")
print(f"Test size  : {meta['test_size']:,}")

### 3.2 Top Features by SHAP Importance

SHAP (SHapley Additive exPlanations) reveals which features drive cancellation predictions the most.

In [ ]:
shap_df = pd.read_csv(os.path.join(cancel_dir, 'shap_feature_importance.csv'))
shap_df.columns = ['feature', 'importance']
top5 = shap_df.head(5)

print("Top 5 Features by SHAP Importance")
print("=" * 45)
for i, row in top5.iterrows():
    bar = '#' * int(row['importance'] / top5['importance'].max() * 30)
    print(f"{row['feature']:30s} {row['importance']:.4f}  {bar}")

### 3.3 Business Impact

Revenue impact was calculated at multiple decision thresholds:

In [ ]:
with open(os.path.join(cancel_dir, 'evaluation_results.json')) as f:
    eval_results = json.load(f)

thresholds = eval_results['business_impact']['threshold_analysis']

print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'Net Revenue Impact':>20}")
print("-" * 55)
for key in ['threshold_0.3', 'threshold_0.4', 'threshold_0.5']:
    t = thresholds[key]
    print(f"{t['threshold']:>10.1f} {t['precision']:>10.4f} {t['recall']:>10.4f} EUR {t['net_revenue_impact']:>14,.2f}")

print(f"\nBest net revenue impact: EUR {thresholds['threshold_0.3']['net_revenue_impact']:,.2f} at threshold 0.3")

### 3.4 Customer Lifetime Value (CLTV)

In [ ]:
with open(os.path.join(MODELS_DIR, 'cltv', 'cltv_summary.json')) as f:
    cltv = json.load(f)

print("CLTV Summary (6-Month Prediction)")
print("=" * 45)
print(f"Total customers        : {cltv['total_customers']:,}")
print(f"Mean CLTV              : EUR {cltv['mean_cltv_6m']:,.2f}")
print(f"Median CLTV            : EUR {cltv['median_cltv_6m']:,.2f}")
print(f"Total predicted value  : EUR {cltv['total_predicted_value_6m']:,.2f}")
print(f"Avg probability alive  : {cltv['avg_prob_alive']:.1%}")
print()
print("Value Segments:")
for seg, val in cltv['segment_summary']['mean'].items():
    count = cltv['segment_summary']['count'][seg]
    print(f"  {seg:15s}  n={count:,}  mean=EUR {val:,.2f}")

### 3.5 Customer Segmentation

In [ ]:
with open(os.path.join(MODELS_DIR, 'segmentation', 'segmentation_summary.json')) as f:
    seg = json.load(f)

print("Customer Segmentation")
print("=" * 45)
print(f"Optimal clusters (k)   : {seg['optimal_k']}")
print(f"Silhouette score       : {seg['silhouette_score']:.4f}")
print(f"Classifier accuracy    : {seg['classifier_accuracy']:.2%}")
print()
for name, count in seg['segment_counts'].items():
    print(f"  {name:20s}  {count:,} customers")

---

## 4. Module 2 — LLM & Unstructured Data

Module 2 applies large language models and NLP to three hotel-industry tasks:

1. **Invoice Classification** — Categorize procurement invoices automatically
2. **Master Item Cleanup** — Fuzzy-match duplicate product names
3. **Review Sentiment Analysis** — Aspect-level sentiment from guest reviews

### 4.1 Invoice Classification — Accuracy Comparison

| Method | Accuracy | Notes |
|---|---|---|
| Rule-Based (keyword matching) | **73.8%** | Fast, no API cost |
| ML Model (TF-IDF + classifier) | **100%** | Trained on labeled data |
| LLM (Groq/Llama) | ~95% | Zero-shot, flexible |

The hybrid pipeline uses rule-based for high-confidence cases and escalates ambiguous invoices to the LLM.

### 4.2 Fuzzy Matching Example

Master item cleanup resolves duplicate product names across hotel procurement systems:

```
Input items:                     Matched master item:
------------------------------------------------------
"Coca Cola 330ml"          -->   "Coca-Cola 330ml"
"Coka Cola 330 ml"         -->   "Coca-Cola 330ml"
"Coca-Cola Can 33cl"       -->   "Coca-Cola 330ml"
"Heineken Beer 500ml"      -->   "Heineken 500ml"
"Heiniken 500 ml"          -->   "Heineken 500ml"
```

The pipeline uses **RapidFuzz** for token-based similarity with configurable thresholds, plus an optional LLM fallback for difficult matches.

### 4.3 Sentiment & Aspect Analysis

In [ ]:
with open(os.path.join(MODELS_DIR, 'review_analysis', 'analysis_summary.json')) as f:
    review = json.load(f)

print("Review Analysis Summary")
print("=" * 50)
print(f"Total reviews analyzed  : {review['total_reviews']:,}")
print(f"Rule-based accuracy     : {review['rule_based_accuracy']:.1%}")
print(f"ML model accuracy       : {review['ml_accuracy']:.1%}")
print()
print("Aspect-Level Models (Cross-Validation):")
print(f"{'Aspect':>15} {'Accuracy':>10} {'Std':>8}")
print("-" * 35)
for aspect, metrics in review['aspect_models'].items():
    print(f"{aspect:>15} {metrics['cv_accuracy']:>10.2%} {metrics['cv_std']:>8.4f}")

---

## 5. Module 3 — Conversational AI

A natural-language analytics chatbot that lets hotel managers ask questions in plain English (or Turkish) and get SQL-backed answers with auto-generated charts.

### Architecture

```
User Question
     |
     v
+------------------+
| Intent Detector  |  Classifies: analytical / general / greeting / ...
+------------------+
     |
     v
+------------------+
| SQL Generator    |  NL --> SQL (with schema context)
+------------------+
     |
     v
+------------------+
| Query Executor   |  Runs SQL on SQLite booking database
+------------------+
     |
     v
+------------------+
| Insight Generator|  LLM summarizes results in natural language
+------------------+
     |
     v
+------------------+
| Chart Generator  |  Auto-selects chart type (bar, line, pie, ...)
+------------------+
```

### LLM Routing Strategy

```
Request --> [Semantic Cache] --hit--> Return cached response
                |                         
               miss                       
                |                         
                v                         
           [Groq API]  --success--> Cache & Return
           (Llama 3)                      
                |                         
             failure                      
                |                         
                v                         
         [Fallback LLM]  --> Return       
```

### Example Q&A Pairs

| User Question | Generated SQL (simplified) | Answer |
|---|---|---|
| "What is the cancellation rate?" | `SELECT AVG(is_canceled) FROM bookings` | ~37% cancellation rate |
| "Which month has the most bookings?" | `SELECT arrival_month, COUNT(*) ... GROUP BY ... ORDER BY ...` | August, with peak summer demand |
| "Average daily rate by hotel type?" | `SELECT hotel, AVG(adr) FROM bookings GROUP BY hotel` | Resort: higher ADR vs. City Hotel |
| "Top 5 countries by revenue?" | `SELECT country, SUM(adr * total_nights) ... LIMIT 5` | PRT, GBR, FRA, ESP, DEU |

---

## 6. Module 4 — MLOps

Production-grade model monitoring and operations.

### 6.1 Data Drift Detection

Uses **Population Stability Index (PSI)** to detect when incoming data distribution shifts from the training baseline.

| PSI Range | Interpretation | Action |
|---|---|---|
| < 0.10 | No significant drift | Monitor |
| 0.10 - 0.25 | Moderate drift | Investigate |
| > 0.25 | Significant drift | Retrain model |

### 6.2 Model Performance Monitoring

Tracks key metrics over time:
- **AUC-ROC** and **F1-Score** for classification models
- **RMSE** for regression tasks
- Automatic comparison against baseline thresholds

### 6.3 Alert System

```
Monitoring Pipeline:

  [Scheduled Check]  -->  [Compute Metrics]  -->  [Compare Thresholds]
                                                        |
                                           +------------+------------+
                                           |                         |
                                        [PASS]                   [ALERT]
                                     Log & Continue          Log + Notify
                                                         (Email / Slack / Log)
```

Alert severity levels: `INFO`, `WARNING`, `CRITICAL`

### 6.4 Health Score

A composite **0-100 health score** per model, combining:
- Data drift magnitude (PSI)
- Prediction performance vs. baseline
- Data quality checks (missing values, outliers)
- Recency of last retraining

---

## 7. Tech Stack

| Category | Technologies | Role |
|---|---|---|
| **ML / Modeling** | XGBoost, LightGBM, CatBoost, Scikit-learn | Classification, regression, clustering |
| **CLTV** | Lifetimes (BG/NBD, Gamma-Gamma) | Customer lifetime value prediction |
| **NLP / LLM** | Groq API (Llama 3), Hugging Face, NLTK | Sentiment, classification, NL-to-SQL |
| **Fuzzy Matching** | RapidFuzz, FuzzyWuzzy | Master item deduplication |
| **Explainability** | SHAP | Feature importance, model transparency |
| **MLOps** | MLflow, custom monitoring | Experiment tracking, drift detection |
| **Data** | Pandas, NumPy, SQLite | Data processing, storage, querying |
| **Visualization** | Plotly, Matplotlib, Seaborn | Interactive charts, dashboards |
| **Dashboard** | Streamlit | 7-page premium web application |
| **Infrastructure** | Docker, Docker Compose | Containerized deployment |
| **Language** | Python 3.11 | End-to-end development |
| **Testing** | Pytest | Unit and integration tests |

---

## 8. Streamlit Premium Dashboard

A 7-page interactive web application with **multi-language support** (English, Turkish, German).

### Page Overview

| # | Page | Description |
|---|---|---|
| 1 | **Cancellation Predictor** | Interactive form to predict booking cancellation probability |
| 2 | **Customer Intelligence** | CLTV segments, action-based cohorts, lifetime value distribution |
| 3 | **Invoice Classifier** | Upload invoices, compare rule-based vs. ML vs. LLM classification |
| 4 | **Item Cleanup** | Fuzzy-match product names against master list |
| 5 | **Review Analyzer** | Paste reviews for sentiment + aspect-level scoring |
| 6 | **Analytics Chatbot** | Ask questions in natural language, get SQL-backed answers + charts |
| 7 | **MLOps Monitor** | Live model health scores, drift metrics, alert history |

### Running the Dashboard

```bash
# From project root
streamlit run app/main.py

# Or with Docker
docker-compose up
```

---

## 9. Project Structure

```
hotel-intelligence-platform/
|
|-- app/                          # Streamlit dashboard
|   |-- main.py                   # Entry point
|   |-- pages/                    # 7 dashboard pages
|   |-- components.py             # Shared UI components
|   |-- i18n.py                   # Multi-language support
|   +-- theme.py                  # Custom styling
|
|-- src/                          # Core source code
|   |-- module_1_predictive/      # Cancellation, CLTV, Segmentation
|   |-- module_2_llm/            # Invoice, Fuzzy Match, Sentiment
|   |-- module_3_conversational/  # Chatbot, NL-to-SQL, LLM Router
|   +-- module_4_mlops/          # Drift, Monitoring, Alerts
|
|-- models/                       # Trained model artifacts
|   |-- cancellation/             # XGBoost + 4 alternatives
|   |-- cltv/                     # BG/NBD + Gamma-Gamma
|   |-- segmentation/             # KMeans + Classifier
|   +-- review_analysis/          # Sentiment + 5 aspect models
|
|-- notebooks/                    # Jupyter notebooks
|-- data/                         # Datasets
|-- tests/                        # Test suite
|-- docs/                         # Documentation
+-- docker-compose.yml            # Container orchestration
```

---

## 10. Conclusion

### Impact Summary

| Area | Achievement |
|---|---|
| Revenue Protection | EUR 2M+ annual savings from cancellation prediction |
| Customer Intelligence | 12K customers scored with 6-month CLTV forecasts |
| Operational Efficiency | Automated invoice classification at 100% ML accuracy |
| Guest Insights | Aspect-level sentiment analysis across 5 hotel dimensions |
| Analytics Democratization | Natural-language querying for non-technical staff |
| Production Readiness | Automated drift detection, alerting, and health scoring |

### Future Improvements

- **Real-time streaming** — Kafka/Spark integration for live booking data
- **A/B testing framework** — Measure actual business impact of model interventions
- **Multi-property support** — Scale platform across hotel chains
- **Advanced LLM agents** — Multi-turn conversation with memory and tool use
- **AutoML pipeline** — Automated model retraining triggered by drift alerts
- **Cloud deployment** — AWS/GCP infrastructure with CI/CD

---

*Built as a comprehensive portfolio project demonstrating end-to-end data science and ML engineering capabilities for the hospitality industry.*